<a href="https://colab.research.google.com/github/incrisvel/adaptaste/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pré-processamento de avaliações textuais de restaurantes em português

In [1]:
%pip -q install nltk spacy pandas emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.1 MB/s eta 0:00:00


In [2]:
import subprocess
import sys

import nltk
import spacy

for resource in ["punkt", "punkt_tab", "stopwords", "rslp"]:
    nltk.download(resource, quiet=True)

try:
    nlp = spacy.load("pt_core_news_sm")
except OSError:
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", "pt_core_news_sm"],
        check=True,
    )
    nlp = spacy.load("pt_core_news_sm")

In [20]:
import pandas as pd
import json

# Abre arquivo de reviews
with open('reviews.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

review_dataset_full = pd.DataFrame(
    review
    for reviews in data.values()
    for review in reviews
)

# Gera um DF com identificação, descrição, data de criação e estabelecimento
review_dataset_text = review_dataset_full[
    review_dataset_full['review_text'].apply(lambda x: 'en' in x)
][[
    'review_id', 'created_date', 'place_id', 'review_text'
]].copy()

# Trata o campo da review, que vem com subpropriedades
review_dataset_text['review_text'] = review_dataset_full['review_text'].apply(
    lambda x: x.get('en', '')
)

In [22]:
import emoji

def remove_emojis(text):
    return emoji.replace_emoji(text, replace='')

review_dataset_text['description_clean'] = (
    review_dataset_text['review_text']
    .apply(remove_emojis)
)

In [23]:
from nltk.tokenize import word_tokenize

# Tokeniza reviews
review_dataset_text['description_tokenized'] = (
    review_dataset_text['description_clean']
    .apply(lambda x: word_tokenize(x, language='portuguese'))
)

In [24]:
# Deixa todas as palavras em minúsculo e remove pontuação
review_dataset_text['description_normalized'] = (
    review_dataset_text['description_tokenized']
    .apply(lambda tokens: [token.casefold() for token in tokens if token.isalpha()])
)

In [25]:
from nltk.corpus import stopwords

preserve_words = {
    # Negação
    "não",
    "nunca",
    "jamais",
    "nem",
    "sem",

    # Intensidade
    "muito",
    "pouco",
    "bastante",
    "demais",

    # Restrição/contraste
    "apenas",
    "só",
    "somente",
    "mas",
    "porém",
    "contudo"
}

stopwords_pt = set(stopwords.words("portuguese"))
stopwords_pt -= preserve_words

# Filtra as stopwords padrão da língua portuguesa
review_dataset_text['description_no_stopwords'] = (
    review_dataset_text['description_normalized']
    .apply(lambda tokens: [token for token in tokens if token not in stopwords_pt])
)

## Redução de Palavras
A fim de comparar o desempenho do modelo com dados pré-processados através de Stemming e Lematização, mantiveram-se as duas técnicas para avaliação posterior. A equipe acredita que o stemming, apesar de reduzir substancialmente o conteúdo das avaliações, tornando-o mais rápido, pode impactar negativamente a interpretação semântica e de sentimento das avaliações. A Lematização em contra partida mantém palavras completas com o seu significado integral.

In [26]:
from nltk.stem import RSLPStemmer

stemmer = RSLPStemmer()

# Reduz as palavras a suas raízes
review_dataset_text['description_stemmed'] = (
    review_dataset_text['description_no_stopwords']
    .apply(lambda tokens: [stemmer.stem(token) for token in tokens])
)

In [27]:
# Padroniza as palavras flexionadas a sua forma canônica. Ex: comidas -> comida
review_dataset_text['description_lemmatized'] = (
    review_dataset_text['description_no_stopwords']
    .apply(
        lambda tokens: [
            token.lemma_
            for token in nlp(" ".join(tokens))
            if token.is_alpha
        ]
    )
)

In [28]:
from nltk.util import ngrams

# Agrupa as reviews em bigramas
review_dataset_text['description_bigrams'] = (
    review_dataset_text['description_lemmatized']
    .apply(lambda tokens: list(ngrams(tokens, 2)) if len(tokens) > 1 else tokens)
)

In [29]:
# Salva o pré-processamento em um arquivo JSON

review_dataset_text.to_json(
    'reviews_preprocessed.json',
    orient='records',
    force_ascii=False,
    indent=2
)